# STAGE 3 — LEAKAGE-SAFE DATA PREPROCESSING
## Explainable Hyperparameter Optimization for Imbalanced Tabular Classification — Bank Marketing

---

## 1. Purpose of Stage 3

Stage 3 is responsible for converting the original UCI Bank Marketing dataset into a form that can safely be used for machine-learning experiments.

The main objective is not simply to "clean" the data. The more important objective is to prepare the data in a way that prevents **data leakage** and maintains a scientifically valid experimental setup.

Our project is a supervised machine-learning problem involving **imbalanced binary classification**.

The dataset contains information about customers who were contacted during a bank marketing campaign. The target variable is `y`, which indicates whether the customer subscribed to a term deposit.

The target contains two classes:

- `no` → customer did not subscribe
- `yes` → customer subscribed

During Stage 1 and Stage 2, we established that the dataset contains:

- 45,211 observations
- 17 original columns
- 16 input features
- 1 target variable
- 7 numerical features
- 9 categorical features
- approximately 88.3% `no`
- approximately 11.7% `yes`

This means the problem is **imbalanced binary classification**.

Stage 3 prepares this dataset for the later stages of our research while protecting the integrity of the experiments.

---

# 2. Why Preprocessing Is Important

Machine-learning algorithms cannot always work directly with raw tabular data.

Our dataset contains both numerical and categorical variables.

For example:

- `age` is numerical.
- `balance` is numerical.
- `campaign` is numerical.
- `job` is categorical.
- `education` is categorical.
- `marital` is categorical.

A machine-learning algorithm needs these inputs to be represented appropriately.

Therefore, we need to perform preprocessing.

However, preprocessing creates an important methodological problem:

> If information from the test data is used while preparing the training data, the final model evaluation can become biased.

This is called **data leakage**.

Therefore, Stage 3 is specifically designed as **leakage-safe preprocessing**.

---

# 3. Cell 1 — Setup and Dataset Loading

The first cell imports the libraries required for Stage 3 and loads the original dataset.

The main libraries used are:

- `pandas`
- `numpy`
- `train_test_split`
- `ColumnTransformer`
- `Pipeline`
- `OneHotEncoder`
- `StandardScaler`

The dataset is loaded from:

`../data/raw/bank-full.csv`

We deliberately use `bank-full.csv` because this is the dataset selected for our current HPO research project.

The dataset contains 45,211 rows and 17 columns.

At this point, the data is still in its original form.

We have not yet:

- separated the target,
- encoded the target,
- split the data,
- scaled numerical features,
- encoded categorical features,
- applied SMOTE,
- trained a model,
- or performed HPO.

This is important because preprocessing should be performed in a controlled sequence.

The first cell also displays the first five rows and the column names so that we can verify that the correct dataset has been loaded.

The expected shape is:

`(45211, 17)`

This means:

- 45,211 rows/observations
- 17 columns

---

# 4. Cell 2 — Separating Features and Target

The second cell separates the dataset into two parts:

- `X` → input features
- `y` → target

We use:

`X = df.drop(columns=["y"])`

This removes the target column from the input features.

Therefore, `X` contains the 16 variables that can be used by the model.

The target is stored separately using:

`y = df["y"].copy()`

The target is currently represented using text:

- `no`
- `yes`

The resulting shapes are expected to be:

`X.shape = (45211, 16)`

and:

`y.shape = (45211,)`

The distinction between `X` and `y` is fundamental to supervised learning.

### What is supervised learning?

Supervised learning means that the model receives:

1. Input information
2. The correct answer associated with that information

The model attempts to learn a relationship between the input variables and the known target.

In our project:

`X` contains customer and campaign information.

`y` contains whether the customer subscribed to a term deposit.

Therefore:

`X → information used for prediction`

`y → answer the model is learning to predict`

Separating `X` and `y` is necessary before train/test splitting and model development.

---

# 5. Cell 3 — Encoding the Target from yes/no to 1/0

The third cell converts the target variable from text labels into numerical labels.

Originally:

- `no`
- `yes`

After encoding:

- `no → 0`
- `yes → 1`

Therefore:

`0 = customer did not subscribe`

`1 = customer subscribed`

This is called **binary target encoding**.

Our problem is called binary classification because there are exactly two possible target classes.

Binary classification can be represented mathematically as:

`y ∈ {0, 1}`

The encoding does not change the meaning of the target. It only changes how the target is represented.

For example:

Original:

`y = yes`

After encoding:

`y = 1`

Both mean exactly the same thing.

The encoding is useful because many machine-learning algorithms and evaluation tools operate naturally with numerical target labels.

A safety check is also included.

When we use:

`y.map({"no": 0, "yes": 1})`

any unexpected target value would become `NaN`.

Therefore, the code checks whether any missing values were created during encoding.

If unexpected values exist, the code raises an error instead of silently continuing.

This is a useful research practice because silent preprocessing errors can produce incorrect experiments.

The expected target distribution remains:

- `0 → 39,922`
- `1 → 5,289`

The class distribution has not changed.

Only the representation has changed.

---

# 6. Cell 4 — Identifying Numerical and Categorical Features

The fourth cell identifies which input features are numerical and which are categorical.

This distinction is important because numerical and categorical variables require different preprocessing techniques.

Our numerical features are:

- `age`
- `balance`
- `day`
- `duration`
- `campaign`
- `pdays`
- `previous`

Therefore, there are 7 numerical features.

Numerical variables contain quantities that can be represented directly as numbers.

For example:

`age = 35`

`balance = 1200`

`campaign = 2`

The categorical features are:

- `job`
- `marital`
- `education`
- `default`
- `housing`
- `loan`
- `contact`
- `month`
- `poutcome`

Therefore, there are 9 categorical features.

Categorical variables represent categories rather than continuous numerical measurements.

For example:

`job = technician`

`marital = married`

`education = secondary`

`contact = cellular`

The total number of features is therefore:

`7 numerical + 9 categorical = 16 features`

This matches the 16 features obtained in Cell 2.

This verification is important because it confirms that no feature was accidentally lost during feature identification.

---

# 7. Cell 5 — Train/Test Split with Stratification

This is one of the most important cells in Stage 3.

The complete dataset is divided into:

- training data
- testing data

We use an 80/20 split.

Approximately:

`80% → training`

`20% → testing`

The training data is used to develop the model.

The test data is held back and used later for final evaluation.

The fundamental principle is:

> The model should not use the test set to learn.

The test set represents unseen data from the perspective of the final model.

---

## Why do we need a test set?

Suppose we train and evaluate the model on exactly the same observations.

The model has already seen those observations during learning.

Therefore, its performance may not represent how it performs on new observations.

The test set gives us a separate collection of observations that the final model did not use for learning.

This allows us to estimate how well the final model generalizes to unseen data.

---

# 8. Why Stratification Is Used

Our target is highly imbalanced.

The approximate distribution is:

`no = 88.3%`

`yes = 11.7%`

The minority class is therefore the `yes` class.

If we performed a purely random split, the class proportions in training and testing could differ slightly.

We therefore use:

`stratify=y`

This tells `train_test_split` to preserve approximately the same target distribution in both sets.

Conceptually:

Original dataset:

`no ≈ 88.3%`

`yes ≈ 11.7%`

Training data:

`no ≈ 88.3%`

`yes ≈ 11.7%`

Testing data:

`no ≈ 88.3%`

`yes ≈ 11.7%`

This is especially important for our research because the minority class is the main challenge being studied.

---

# 9. Why random_state=42 Is Used

The split also uses:

`random_state=42`

A machine-learning train/test split contains a random component.

If we run the same code without fixing the random state, the observations assigned to training and testing may change between runs.

Using a fixed random state makes the experiment reproducible.

Reproducibility means that another researcher, or we ourselves at a later point, can repeat the experiment using the same configuration and obtain the same split.

The number `42` itself has no scientific meaning.

It is simply a fixed seed.

The important concept is not the number 42; the important concept is using a fixed random seed consistently.

---

# 10. Cell 6 — Understanding Data Leakage

Cell 6 is primarily a conceptual cell.

It explains one of the most important methodological issues in our research:

**data leakage**.

Data leakage occurs when information that should be unavailable to the model during training influences the training process.

In simple terms:

> The model receives information it should not have access to.

This can make model performance appear better than it really is.

For a research paper, this is a serious problem because leaked experiments can produce misleading performance measurements.

---

# 11. Example of Scaling Leakage

Suppose we want to standardize `age`.

The correct approach is:

Training data:

`X_train`

↓

Learn mean and standard deviation

↓

Transform training data

The test data is then transformed using the same parameters:

`X_test`

↓

Use training mean and standard deviation

↓

Transform test data

The incorrect approach is to combine training and testing data and fit the scaler on everything.

That would allow the distribution of the test data to influence the preprocessing parameters.

The test set would no longer be completely unseen.

Therefore, our methodology follows:

> Fit preprocessing using training data only.

---

# 12. Example of One-Hot Encoding Leakage

Categorical preprocessing can also cause leakage if handled incorrectly.

Suppose training data contains:

- admin
- technician
- management

but the test set contains another category that was not observed during training.

Our encoder should learn its category structure from training data.

We use:

`OneHotEncoder(handle_unknown="ignore")`

This means that if an unseen category appears during transformation, the encoder will handle it safely rather than failing.

The important principle is that the encoder itself is fitted using the training data.

The test data is only transformed using the learned encoder.

---

# 13. Example of SMOTE Leakage

SMOTE is particularly important in our research because our project focuses on imbalanced classification.

SMOTE stands for:

**Synthetic Minority Over-sampling Technique**

It creates synthetic minority-class observations to help a model learn from an underrepresented class.

However, SMOTE must not be applied to the complete dataset before the train/test split.

Incorrect:

`Entire dataset`

↓

`SMOTE`

↓

`Train/Test Split`

This can allow information from observations that later become part of the test set to influence synthetic training observations.

Correct:

`Original dataset`

↓

`Train/Test Split`

↓

`Training data`

↓

`SMOTE`

↓

`Model training`

The situation becomes even more important during cross-validation.

For each cross-validation fold, SMOTE should operate only on the training portion of that fold.

Therefore, later our research pipeline will follow the conceptual structure:

`Training fold`

↓

`Preprocessing`

↓

`SMOTE`

↓

`Model`

↓

`Validation fold`

The validation fold remains untouched by SMOTE.

This is a major methodological requirement for our imbalance-aware HPO experiments.

---

# 14. Cell 7 — Numerical Preprocessing Using StandardScaler

Cell 7 creates the preprocessing component for numerical features.

We use:

`StandardScaler`

StandardScaler performs standardization.

The basic formula is:

`z = (x - μ) / σ`

where:

- `x` = original feature value
- `μ` = mean of the feature
- `σ` = standard deviation of the feature
- `z` = standardized value

After standardization, a numerical feature is generally centered around zero and has a standard deviation close to one.

For example, suppose a feature has:

`mean = 100`

and:

`standard deviation = 20`

An original value of:

`x = 140`

would become:

`z = (140 - 100) / 20`

`z = 2`

The original value 140 is therefore represented as approximately 2 standard deviations above the mean.

---

# 15. Why Numerical Scaling Is Useful

Different numerical features in our dataset can have very different ranges.

For example:

`age` may be around tens.

`balance` can be in thousands.

`duration` can be hundreds or thousands.

`campaign` usually has a much smaller range.

For algorithms that are sensitive to feature magnitude, scaling can be useful.

However, it is important not to make the incorrect statement that every machine-learning algorithm requires scaling.

Tree-based algorithms such as Random Forest generally do not require standardization in the same way that linear or distance-based algorithms may.

Our preprocessing architecture is therefore modular.

Later, depending on the model family used in the baseline and HPO experiments, we can make appropriate preprocessing decisions.

---

# 16. Why StandardScaler Is Inside a Pipeline

Instead of manually calculating scaling parameters, we create:

`numerical_transformer`

using a `Pipeline`.

A pipeline is a sequence of operations that are executed in a controlled order.

This becomes especially useful later when we combine:

`Preprocessing`

↓

`Imbalance handling`

↓

`Model`

↓

`Hyperparameter optimization`

↓

`Cross-validation`

The pipeline structure helps prevent accidental leakage because sklearn can fit each step using the appropriate training portion.

---

# 17. Cell 8 — Categorical Preprocessing Using OneHotEncoder

Cell 8 creates the preprocessing component for categorical features.

We use:

`OneHotEncoder`

Categorical variables cannot normally be supplied directly to many machine-learning algorithms as strings.

For example:

`marital = married`

cannot simply be treated as the numerical value 2.

Doing this would incorrectly introduce an ordering.

For example, assigning:

`divorced = 0`

`married = 1`

`single = 2`

would suggest that:

`single > married > divorced`

in a mathematical sense.

There is no such numerical ordering in the original variable.

One-hot encoding avoids this problem.

---

# 18. How One-Hot Encoding Works

Suppose `marital` contains:

- divorced
- married
- single

One-hot encoding can create:

- `marital_divorced`
- `marital_married`
- `marital_single`

A married customer would be represented approximately as:

`0, 1, 0`

A single customer would be:

`0, 0, 1`

A divorced customer would be:

`1, 0, 0`

Each category receives its own binary indicator.

This converts categorical information into a numerical representation without falsely introducing an ordinal relationship.

---

# 19. Why handle_unknown="ignore" Is Used

We use:

`handle_unknown="ignore"`

This is important when transforming future observations.

Suppose a category was not observed when the encoder was fitted.

If the encoder is configured incorrectly, encountering that category later may produce an error.

With:

`handle_unknown="ignore"`

the encoder safely handles an unseen category.

This makes the preprocessing pipeline more robust.

It is also important for train/test processing because the encoder is fitted on training data and then used to transform the test data.

---

# 20. Treatment of "unknown" Values

During Stage 1 and Stage 2, we discovered that some categorical columns contain the literal category:

`unknown`

Examples include:

- `job`
- `education`
- `contact`
- `poutcome`

We decided **not to remove these observations**.

This is an important project decision.

We are not treating the string `"unknown"` as the same thing as a missing `NaN` value.

Instead, it is retained as an explicit category.

Therefore, if:

`contact = unknown`

the record remains in the dataset.

The OneHotEncoder will treat `"unknown"` as one of the categories.

This means our preprocessing does not unnecessarily discard observations simply because a categorical value is recorded as `"unknown"`.

---

# 21. Cell 9 — Combining Numerical and Categorical Processing

At this point we have two preprocessing branches:

Numerical:

`Numerical features`

↓

`StandardScaler`

Categorical:

`Categorical features`

↓

`OneHotEncoder`

We need to combine these branches into one preprocessing system.

We therefore use:

`ColumnTransformer`

ColumnTransformer allows us to specify which transformation should be applied to which columns.

Our configuration is:

`num → numerical_transformer → numerical_columns`

and:

`cat → categorical_transformer → categorical_columns`

Therefore:

7 numerical features

↓

StandardScaler

and:

9 categorical features

↓

OneHotEncoder

The outputs are then combined into a single transformed feature representation.

---

# 22. What remainder="drop" Means

The ColumnTransformer uses:

`remainder="drop"`

This means:

> Any column that is not explicitly assigned to a transformer will be dropped.

In our current dataset, all 16 features are explicitly assigned:

7 numerical

+

9 categorical

=

16 total features

Therefore, no intended feature should be dropped.

Using `remainder="drop"` also makes the preprocessing definition explicit.

We do not want an accidentally omitted feature to silently pass into the model without preprocessing.

---

# 23. Cell 10 — Creating the Complete Preprocessing Pipeline

Cell 10 places the ColumnTransformer inside a complete sklearn Pipeline.

The resulting object is:

`preprocessing_pipeline`

The pipeline currently contains:

`preprocessor`

The key operation is:

`fit_transform(X_train)`

This performs two operations:

1. `fit`
2. `transform`

---

# 24. What Does Fit Mean?

When a preprocessing object is fitted, it learns information required to perform the transformation.

For StandardScaler, fitting learns:

- mean
- standard deviation

for each numerical feature.

For OneHotEncoder, fitting learns:

- the categories observed in the training data

Therefore, fitting is a learning operation.

This is why it is important that fitting occurs only on the training data.

---

# 25. What Does Transform Mean?

Transform means:

> Apply the already learned preprocessing rules to data.

For training:

`fit_transform(X_train)`

means:

`learn preprocessing parameters from training data`

and then:

`transform training data`

For testing:

`transform(X_test)`

means:

`do not learn anything new`

and simply:

`apply the training preprocessing to the test data`

This distinction is one of the most important concepts in our entire preprocessing stage.

---

# 26. Why We Never Use fit_transform(X_test)

We should never do:

`preprocessing_pipeline.fit_transform(X_test)`

for final evaluation.

Doing so would make the preprocessing learn information from the test data.

Instead, we use:

`preprocessing_pipeline.transform(X_test)`

The test data is therefore treated as unseen data.

This maintains the integrity of the final evaluation.

---

# 27. Why the Number of Features Increases

Before preprocessing:

`16 features`

After one-hot encoding:

`more than 16 features`

This is normal.

Categorical variables can contain multiple categories.

For example, `job` may contain many different job categories.

One-hot encoding converts those categories into separate columns.

Therefore, one original categorical column can become many numerical columns.

The exact number of transformed features depends on the categories learned from the training data.

---

# 28. Cell 11 — Inspecting the Transformed Features

Cell 11 is used to inspect what the preprocessing system actually produced.

The function:

`get_feature_names_out()`

allows us to retrieve the names of the transformed features.

For example:

`num__age`

indicates that `age` came from the numerical preprocessing branch.

A name such as:

`cat__job_admin.`

indicates that the feature was created from the categorical `job` variable.

This is useful because after one-hot encoding, the original feature structure changes.

Instead of having:

`job = technician`

we may now have separate indicator features representing job categories.

The transformed data is now numerical and can be supplied to machine-learning algorithms.

---

# 29. Why Only a Small Sample Is Displayed

One-hot encoding can create a relatively large feature matrix.

Converting the entire transformed dataset into a dense DataFrame can consume unnecessary memory.

Therefore, Cell 11 inspects only the first five processed observations.

This is sufficient to verify that:

- numerical values have been transformed,
- categorical variables have been encoded,
- feature names are present,
- the transformed structure is correct.

This is a practical memory-conscious approach.

---

# 30. Sparse Matrices

OneHotEncoder can produce a sparse representation.

A sparse matrix stores mainly the values that are non-zero.

This is efficient because one-hot encoded data contains many zeros.

For example:

A row might have hundreds of possible categorical indicator columns, but only a small number of them will contain `1`.

Instead of storing every zero explicitly, a sparse representation can store only the meaningful non-zero values.

This can significantly reduce memory usage.

Therefore, Cell 11 avoids unnecessarily converting the entire processed matrix into a dense representation.

---

# 31. Cell 12 — Preprocessing Validation

Cell 12 performs automated consistency checks.

This is an important research practice.

A successful Python execution does not automatically mean that the methodology is correct.

We therefore explicitly verify several conditions.

---

## Check 1 — Training Rows

The number of processed training observations must equal the number of target values in `y_train`.

Therefore:

`X_train_processed rows = y_train length`

If these do not match, the model would not know which target belongs to which observation.

---

## Check 2 — Testing Rows

The same principle applies to testing:

`X_test_processed rows = y_test length`

The number of observations and target labels must match.

---

## Check 3 — Same Number of Features

The processed training and testing datasets must have the same number of feature columns.

Therefore:

`X_train_processed.shape[1] = X_test_processed.shape[1]`

This is essential because a model trained on one feature structure cannot normally be evaluated on a different feature structure.

---

## Check 4 — Valid Target Values

We verify that the training and testing target values are only:

`0`

and:

`1`

This confirms that the binary target encoding remains correct.

---

## Check 5 — Missing Values

We also check the processed data for NaN values.

Unexpected NaN values could cause problems during model training.

The validation code handles both sparse and dense matrices.

This avoids unnecessarily converting a potentially large sparse matrix into a dense matrix.

---

# 32. Why Automated Assertions Are Useful

The code uses statements such as:

`assert condition`

An assertion means:

> The researcher expects this condition to be true.

If the condition is false, Python stops and reports the problem.

For example:

If the training and testing feature counts were accidentally different, the assertion would stop the notebook instead of allowing the error to propagate into later experiments.

This is useful for research reproducibility and debugging.

---

# 33. Cell 13 — Saving the Train/Test Split

Cell 13 saves the exact train/test assignment.

The file is:

`train_test_split_indices.csv`

It contains:

- original dataset index
- split assignment

For example:

`dataset_index = 10`

`split = train`

or:

`dataset_index = 25`

`split = test`

The purpose is reproducibility.

Our research will involve multiple experiments.

We want those experiments to be comparable.

If one experiment uses one random split and another experiment uses a different split, the difference in performance could partly be caused by the different data partition rather than by the modeling technique.

By preserving the split information, we can maintain a consistent experimental foundation.

---

# 34. Why We Are Not Saving the Processed Matrix as the Main Research Dataset

We created:

`X_train_processed`

and:

`X_test_processed`

during Stage 3.

These are useful for inspecting and validating preprocessing.

However, they should not become the only data representation used for our later HPO experiments.

This is because our HPO experiments will involve cross-validation.

During cross-validation, preprocessing should ideally be fitted separately within each training fold.

For example:

`Fold training data`

↓

`Fit preprocessing`

↓

`Transform fold training data`

↓

`Apply imbalance strategy`

↓

`Train model`

↓

`Evaluate on validation fold`

This prevents information from one validation fold from influencing preprocessing learned for another fold.

Therefore, the reusable preprocessing object itself is more important than the already-transformed matrix.

---

# 35. Cell 14 — Final Stage Summary

Cell 14 prints a complete summary of the decisions made during Stage 3.

It confirms:

- dataset size
- number of features
- target encoding
- numerical features
- categorical features
- train/test split
- random state
- stratification
- numerical preprocessing
- categorical preprocessing
- treatment of unknown values
- processed dimensions
- leakage protection
- reproducibility information

This final summary acts as a checkpoint before moving into the next research stage.

---

# 36. Final Stage 3 Methodology

Our complete preprocessing methodology is:

`Original UCI Bank Marketing Dataset`

↓

`Separate X and y`

↓

`Encode target`

`no → 0`

`yes → 1`

↓

`Stratified 80/20 train-test split`

↓

`Training data`

`Test data`

↓

`Numerical features`

`StandardScaler`

↓

`Categorical features`

`OneHotEncoder(handle_unknown="ignore")`

↓

`ColumnTransformer`

↓

`Leakage-safe preprocessing pipeline`

↓

`Validation checks`

↓

`Save split information`

---

# 37. Important Decisions Made in Stage 3

### Decision 1 — Keep all observations

We are not removing observations simply because a categorical field contains `"unknown"`.

The `"unknown"` category is retained.

---

### Decision 2 — Do not remove duplicates unless verified

Stage 1 established the dataset quality, and we are not introducing unnecessary row removal during preprocessing.

---

### Decision 3 — Do not apply SMOTE yet

SMOTE belongs to the imbalance-handling stage.

It should not be applied during this preprocessing stage.

More importantly, it must never be applied to the complete dataset before train/test splitting.

---

### Decision 4 — Keep the test set isolated

The test set is reserved for final evaluation.

We should not use test performance to repeatedly select the best hyperparameters.

---

### Decision 5 — Use stratification

Because our problem is imbalanced, the train/test split preserves approximately the same class distribution.

---

### Decision 6 — Use reproducible splitting

`random_state=42` ensures that our primary train/test split can be reproduced.

---

### Decision 7 — Use a pipeline architecture

Preprocessing is represented as reusable sklearn objects rather than a collection of unrelated manual transformations.

This will be important when we integrate:

- cross-validation
- SMOTE
- hyperparameter optimization
- multiple classification models
- threshold optimization

---

# 38. Relationship Between Stage 3 and Our Research Question

Our research is not simply:

> "Can we predict whether a bank customer subscribes?"

Our broader research question concerns how **hyperparameter optimization and imbalance-aware modeling affect binary classification performance**, while maintaining explainability and methodological reliability.

Therefore, preprocessing must provide a trustworthy foundation.

If preprocessing leaks information, then later comparisons between:

- baseline models,
- imbalance techniques,
- HPO strategies,
- multi-objective optimization,
- threshold optimization,

could become unreliable.

Therefore, leakage-safe preprocessing is not just a coding requirement.

It is part of the **research methodology**.

---

# 39. Relationship Between Stage 3 and Imbalanced Classification

Our target distribution is approximately:

`No → 88.3%`

`Yes → 11.7%`

The positive subscription class is therefore the minority class.

A model could obtain high accuracy simply by predicting the majority class very frequently.

For example, a hypothetical classifier that predicted `no` for every customer could appear to achieve approximately 88% accuracy while identifying no actual subscribers.

This demonstrates why accuracy alone will not be sufficient for our research.

Later we will evaluate metrics such as:

- Precision
- Recall
- F1-score
- Balanced Accuracy
- ROC-AUC
- PR-AUC

These metrics allow us to examine minority-class behavior more carefully.

---

# 40. Why Stage 3 Must Come Before HPO

Hyperparameter optimization searches for model configurations that improve a selected objective.

For example, HPO might search over:

- tree depth
- number of estimators
- learning rate
- regularization parameters
- minimum samples per leaf
- class weights
- other model-specific parameters

However, if the data pipeline itself is flawed, HPO may optimize the wrong experimental setup.

A powerful optimizer cannot fix data leakage.

Therefore:

`Correct data preparation`

must come before:

`Reliable model optimization`

This is why Stage 3 is an important foundation for the later HPO experiments.

---

# 41. What We Have NOT Done Yet

Stage 3 does NOT perform:

- model training
- baseline comparison
- SMOTE
- RandomUnderSampler
- class-weight experiments
- hyperparameter optimization
- threshold optimization
- SHAP explainability
- multi-objective optimization
- robustness analysis

Those are later stages.

This separation is intentional.

We first establish a reliable preprocessing foundation.

Then we can systematically study the effect of the later techniques.

---

# 42. Stage 3 Final Conclusion

Stage 3 successfully converts the raw Bank Marketing dataset into a structured machine-learning-ready representation while maintaining a leakage-safe experimental design.

The dataset is separated into input features and target labels.

The target is converted from `yes/no` into `1/0`.

Numerical and categorical features are identified separately.

The dataset is divided into training and testing sets using an 80/20 stratified split.

Numerical features are prepared using StandardScaler.

Categorical features are prepared using OneHotEncoder with `handle_unknown="ignore"`.

The literal `"unknown"` category is retained rather than removed.

The preprocessing components are combined using ColumnTransformer and Pipeline.

The preprocessing is fitted using training data and applied to test data without fitting on the test set.

Validation checks confirm that the resulting datasets have compatible dimensions and valid target values.

The exact train/test split is also saved to support reproducibility.

Most importantly, the methodology establishes the foundation for our later **imbalance-aware HPO experiments without prematurely applying SMOTE or allowing test information to influence training**.

---

# 43. Research Pipeline After Stage 3

At the end of Stage 3, our overall research pipeline is:

`UCI Bank Marketing Dataset`

↓

`Data Understanding`

↓

`EDA and Imbalance Analysis`

↓

`Leakage-Safe Preprocessing`

↓

**Stage 4 — Baseline Models**

↓

**Stage 5 — Imbalance Strategies**

↓

**Stage 6 — Hyperparameter Optimization**

↓

**Stage 7 — Multi-Objective HPO**

↓

**Stage 8 — Threshold Optimization**

↓

**Stage 9 — Robustness Analysis**

↓

**Stage 10 — SHAP Explainability**

↓

**Stage 11 — Explanation Stability**

↓

**Stage 12 — Final Results and Research Conclusions**

The purpose of this staged structure is to make every experimental improvement measurable.

We should always be able to answer:

> Did the improvement come from preprocessing, imbalance handling, hyperparameter optimization, objective selection, threshold optimization, or another methodological component?

That separation is essential for turning the project from a simple machine-learning implementation into a structured experimental research study.

In [1]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 1: Setup and Dataset Loading
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Dataset path
data_path = "../data/raw/bank-full.csv"

# Load the original dataset
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Dataset shape: (45211, 17)

First 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,18,student,single,primary,no,1944,no,no,telephone,10,aug,122,3,-1,0,unknown,no
1,18,student,single,unknown,no,108,no,no,cellular,10,aug,167,1,-1,0,unknown,yes
2,18,student,single,primary,no,608,no,no,cellular,12,aug,267,1,-1,0,unknown,yes
3,18,student,single,unknown,no,35,no,no,telephone,21,aug,104,2,-1,0,unknown,no
4,18,student,single,secondary,no,5,no,no,cellular,24,aug,143,2,-1,0,unknown,no



Columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


In [2]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 2: Separate Features and Target
# ============================================

# X = input features
# Remove the target column "y" from the dataset
X = df.drop(columns=["y"]).copy()

# y = target variable
# Keep only the target column
y = df["y"].copy()

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

print("\nNumber of feature columns:", X.shape[1])

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget values and counts:")
print(y.value_counts())

Features (X) shape: (45211, 16)
Target (y) shape: (45211,)

Number of feature columns: 16

Feature columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']

Target values and counts:
y
no     39922
yes     5289
Name: count, dtype: int64


In [3]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 3: Encode Target yes/no → 1/0
# ============================================

# Convert the target labels into binary numerical values
y = y.map({
    "no": 0,
    "yes": 1
})

# Safety check:
# If any value could not be converted, it becomes NaN.
if y.isna().any():
    raise ValueError("Target encoding produced missing values. Check the original target labels.")

print("Target encoding completed successfully!")

print("\nTarget values and counts:")
print(y.value_counts().sort_index())

print("\nTarget data type:")
print(y.dtype)

print("\nUnique target values:")
print(sorted(y.unique()))

Target encoding completed successfully!

Target values and counts:
y
0    39922
1     5289
Name: count, dtype: int64

Target data type:
int64

Unique target values:
[np.int64(0), np.int64(1)]


In [4]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 4: Identify Numerical and Categorical Features
# ============================================

# Identify numerical columns
numerical_columns = X.select_dtypes(include=np.number).columns.tolist()

# Identify categorical/text columns
categorical_columns = X.select_dtypes(include="object").columns.tolist()

print("Numerical features:")
print(numerical_columns)

print("\nNumber of numerical features:", len(numerical_columns))

print("\nCategorical features:")
print(categorical_columns)

print("\nNumber of categorical features:", len(categorical_columns))

print("\nTotal features identified:",
      len(numerical_columns) + len(categorical_columns))

Numerical features:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Number of numerical features: 7

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Number of categorical features: 9

Total features identified: 16


In [5]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 5: Train/Test Split with Stratification
# ============================================

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test split completed successfully!")

print("\nTraining data:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nTesting data:")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTraining target percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts())
print("\nTesting target percentage:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train/Test split completed successfully!

Training data:
X_train shape: (36168, 16)
y_train shape: (36168,)

Testing data:
X_test shape: (9043, 16)
y_test shape: (9043,)

Training target distribution:
y
0    31937
1     4231
Name: count, dtype: int64

Training target percentage:
y
0    88.3
1    11.7
Name: proportion, dtype: float64

Testing target distribution:
y
0    7985
1    1058
Name: count, dtype: int64

Testing target percentage:
y
0    88.3
1    11.7
Name: proportion, dtype: float64


In [7]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 7: Numerical Preprocessing
# ============================================

# Create a preprocessing pipeline for numerical features
numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

print("Numerical preprocessing pipeline created successfully!")

print("\nNumerical features that will be scaled:")
print(numerical_columns)

print("\nNumber of numerical features:",
      len(numerical_columns))

Numerical preprocessing pipeline created successfully!

Numerical features that will be scaled:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Number of numerical features: 7


In [8]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 8: Categorical Preprocessing
# ============================================

# Create a preprocessing pipeline for categorical features
categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

print("Categorical preprocessing pipeline created successfully!")

print("\nCategorical features that will be one-hot encoded:")
print(categorical_columns)

print("\nNumber of categorical features:",
      len(categorical_columns))

print("\nEncoder setting:")
print("handle_unknown = 'ignore'")

Categorical preprocessing pipeline created successfully!

Categorical features that will be one-hot encoded:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Number of categorical features: 9

Encoder setting:
handle_unknown = 'ignore'


In [9]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 9: Combine Numerical and Categorical
# ============================================

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_columns
        ),
        (
            "cat",
            categorical_transformer,
            categorical_columns
        )
    ],
    remainder="drop"
)

print("Combined preprocessing object created successfully!")

print("\nNumerical preprocessing:")
print("StandardScaler →", len(numerical_columns), "features")

print("\nCategorical preprocessing:")
print("OneHotEncoder →", len(categorical_columns), "features")

print("\nColumnTransformer is ready.")

Combined preprocessing object created successfully!

Numerical preprocessing:
StandardScaler → 7 features

Categorical preprocessing:
OneHotEncoder → 9 features

ColumnTransformer is ready.


In [10]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 10: Build and Apply Preprocessing Pipeline
# ============================================

# Create the complete preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

print("Preprocessing pipeline created successfully!")

# Fit the preprocessing pipeline ONLY on training data
# and transform the training data
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Transform the test data using the preprocessing
# already learned from the training data
X_test_processed = preprocessing_pipeline.transform(X_test)

print("\nPreprocessing completed successfully!")

print("\nProcessed training data shape:")
print(X_train_processed.shape)

print("\nProcessed testing data shape:")
print(X_test_processed.shape)

Preprocessing pipeline created successfully!

Preprocessing completed successfully!

Processed training data shape:
(36168, 51)

Processed testing data shape:
(9043, 51)


In [11]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 11: Inspect Transformed Features
# ============================================

# Get the names of all transformed features
feature_names = (
    preprocessing_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("Number of transformed features:", len(feature_names))

print("\nFirst 20 transformed feature names:")
print(feature_names[:20])

print("\nProcessed training data type:")
print(type(X_train_processed))

print("\nProcessed training data shape:")
print(X_train_processed.shape)

print("\nProcessed testing data shape:")
print(X_test_processed.shape)


# Inspect only the first 5 processed training rows
sample_processed = X_train_processed[:5]

# If the result is a sparse matrix, convert only these 5 rows
# to a normal NumPy array for easy viewing.
if hasattr(sample_processed, "toarray"):
    sample_processed = sample_processed.toarray()

processed_sample_df = pd.DataFrame(
    sample_processed,
    columns=feature_names
)

print("\nFirst 5 processed training observations:")
display(processed_sample_df)

Number of transformed features: 51

First 20 transformed feature names:
['num__age' 'num__balance' 'num__day' 'num__duration' 'num__campaign'
 'num__pdays' 'num__previous' 'cat__job_admin.' 'cat__job_blue-collar'
 'cat__job_entrepreneur' 'cat__job_housemaid' 'cat__job_management'
 'cat__job_retired' 'cat__job_self-employed' 'cat__job_services'
 'cat__job_student' 'cat__job_technician' 'cat__job_unemployed'
 'cat__job_unknown' 'cat__marital_divorced']

Processed training data type:
<class 'numpy.ndarray'>

Processed training data shape:
(36168, 51)

Processed testing data shape:
(9043, 51)

First 5 processed training observations:


,num__age,num__balance,num__day,num__duration,num__campaign,num__pdays,num__previous,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,...,cat__month_jun,cat__month_mar,cat__month_may,cat__month_nov,cat__month_oct,cat__month_sep,cat__poutcome_failure,cat__poutcome_other,cat__poutcome_success,cat__poutcome_unknown
0,0.005612,-0.344452,-0.216345,-0.895275,0.722821,-0.411234,-0.241666,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1.324274,0.220373,-1.658186,-0.189926,-0.245092,2.319256,0.174096,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-1.030479,1.376263,0.384423,0.986954,-0.567729,-0.411234,-0.241666,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.276958,-0.417925,0.384423,0.912911,2.013372,-0.411234,-0.241666,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.005612,-0.363804,-1.538032,-0.458816,1.045459,-0.411234,-0.241666,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [12]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 12: Preprocessing Validation Checks
# ============================================

print("Running preprocessing validation checks...\n")

# ------------------------------------------------
# 1. Check number of rows
# ------------------------------------------------

assert X_train_processed.shape[0] == len(y_train), \
    "Mismatch between processed training rows and y_train."

assert X_test_processed.shape[0] == len(y_test), \
    "Mismatch between processed testing rows and y_test."


# ------------------------------------------------
# 2. Check number of processed features
# ------------------------------------------------

assert X_train_processed.shape[1] == X_test_processed.shape[1], \
    "Training and testing have different numbers of processed features."


# ------------------------------------------------
# 3. Check feature names
# ------------------------------------------------

assert X_train_processed.shape[1] == len(feature_names), \
    "Number of feature names does not match processed feature count."


# ------------------------------------------------
# 4. Check target values
# ------------------------------------------------

assert set(y_train.unique()).issubset({0, 1}), \
    "Unexpected target values found in y_train."

assert set(y_test.unique()).issubset({0, 1}), \
    "Unexpected target values found in y_test."


# ------------------------------------------------
# 5. Check for missing values
# ------------------------------------------------

# For sparse matrices, checking .data avoids
# unnecessarily converting the entire matrix to dense form.
if hasattr(X_train_processed, "data"):
    train_nan_count = np.isnan(X_train_processed.data).sum()
    test_nan_count = np.isnan(X_test_processed.data).sum()
else:
    train_nan_count = np.isnan(X_train_processed).sum()
    test_nan_count = np.isnan(X_test_processed).sum()


assert train_nan_count == 0, \
    "Missing/NaN values found in processed training data."

assert test_nan_count == 0, \
    "Missing/NaN values found in processed testing data."


# ------------------------------------------------
# 6. Print final validation information
# ------------------------------------------------

print("All preprocessing validation checks passed successfully!\n")

print("Training rows:", X_train_processed.shape[0])
print("Testing rows:", X_test_processed.shape[0])

print("Processed features:", X_train_processed.shape[1])

print("y_train rows:", len(y_train))
print("y_test rows:", len(y_test))

print("NaN values in processed training data:", train_nan_count)
print("NaN values in processed testing data:", test_nan_count)

print("\nPreprocessing output is consistent and ready for the next stage.")

Running preprocessing validation checks...

All preprocessing validation checks passed successfully!

Training rows: 36168
Testing rows: 9043
Processed features: 51
y_train rows: 36168
y_test rows: 9043
NaN values in processed training data: 0
NaN values in processed testing data: 0

Preprocessing output is consistent and ready for the next stage.


In [13]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 13: Save Train/Test Split Information
# ============================================

# Create a table containing the original dataset index
# and whether that observation belongs to train or test.

split_info = pd.DataFrame({
    "dataset_index": X.index,
    "split": np.where(
        X.index.isin(X_train.index),
        "train",
        "test"
    )
})

# Create the output directory if it does not already exist
import os

os.makedirs("../results/metrics", exist_ok=True)

# Save the split information
split_path = "../results/metrics/train_test_split_indices.csv"

split_info.to_csv(
    split_path,
    index=False
)

print("Train/test split information saved successfully!")

print("\nFile saved at:")
print(split_path)

print("\nSaved rows:", len(split_info))

print("\nSplit counts:")
print(split_info["split"].value_counts())

print("\nFirst 10 saved records:")
display(split_info.head(10))

Train/test split information saved successfully!

File saved at:
../results/metrics/train_test_split_indices.csv

Saved rows: 45211

Split counts:
split
train    36168
test      9043
Name: count, dtype: int64

First 10 saved records:


,dataset_index,split
0,0,train
1,1,test
2,2,train
3,3,train
4,4,train
5,5,train
6,6,test
7,7,train
8,8,train
9,9,train


In [14]:
# ============================================
# STAGE 3 — LEAKAGE-SAFE PREPROCESSING
# Cell 14: Final Stage Summary
# ============================================

print("=" * 60)
print("STAGE 3 — PREPROCESSING SUMMARY")
print("=" * 60)

print("\n1. DATASET")
print("-" * 60)
print("Original dataset rows:", len(df))
print("Original dataset columns:", df.shape[1])

print("\n2. FEATURES AND TARGET")
print("-" * 60)
print("Total input features:", X.shape[1])
print("Target column: y")
print("Target encoding: no → 0, yes → 1")

print("\n3. FEATURE TYPES")
print("-" * 60)
print("Numerical features:", len(numerical_columns))
print("Categorical features:", len(categorical_columns))

print("\nNumerical:")
print(numerical_columns)

print("\nCategorical:")
print(categorical_columns)

print("\n4. TRAIN / TEST SPLIT")
print("-" * 60)
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Test size:", "20%")
print("Random state:", 42)
print("Stratification:", "Yes")

print("\n5. NUMERICAL PREPROCESSING")
print("-" * 60)
print("Method:", "StandardScaler")

print("\n6. CATEGORICAL PREPROCESSING")
print("-" * 60)
print("Method:", "OneHotEncoder")
print("Unknown categories:", "handle_unknown='ignore'")
print("'unknown' values in original dataset:", "Retained")

print("\n7. PROCESSED DATA")
print("-" * 60)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Processed feature count:", len(feature_names))

print("\n8. DATA LEAKAGE PROTECTION")
print("-" * 60)
print("Preprocessing fitted on training data only: Yes")
print("Test data used to fit preprocessing: No")
print("SMOTE applied in Stage 3: No")

print("\n9. REPRODUCIBILITY")
print("-" * 60)
print("Random state:", 42)
print("Train/test split saved:", "Yes")
print("Saved file:", split_path)

print("\n" + "=" * 60)
print("STAGE 3 COMPLETED SUCCESSFULLY")
print("=" * 60)

STAGE 3 — PREPROCESSING SUMMARY

1. DATASET
------------------------------------------------------------
Original dataset rows: 45211
Original dataset columns: 17

2. FEATURES AND TARGET
------------------------------------------------------------
Total input features: 16
Target column: y
Target encoding: no → 0, yes → 1

3. FEATURE TYPES
------------------------------------------------------------
Numerical features: 7
Categorical features: 9

Numerical:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Categorical:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

4. TRAIN / TEST SPLIT
------------------------------------------------------------
Training observations: 36168
Testing observations: 9043
Test size: 20%
Random state: 42
Stratification: Yes

5. NUMERICAL PREPROCESSING
------------------------------------------------------------
Method: StandardScaler

6. CATEGORICAL PREPROCESSING
---------------------------